In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Detect environment
if Path("/kaggle/input").exists():
    DATA_DIR = Path("/kaggle/input/ieee-fraud-detection")
else:
    DATA_DIR = Path("data/raw")

print("Using data directory:", DATA_DIR)

for file_path in DATA_DIR.iterdir():
    print(file_path)

Using data directory: data\raw
data\raw\test_identity.csv
data\raw\test_transaction.csv
data\raw\train_identity.csv
data\raw\train_transaction.csv


# Dagshub/Mlflow initialization

In [2]:
pip install dagshub mlflow scikit-learn pandas matplotlib seaborn skops --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import dagshub
import mlflow
import mlflow.sklearn

dagshub.init(repo_owner='myvari', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

Accessing as myvari

Initialized MLflow to track repo "myvari/IEEE-CIS-Fraud-Detection"

Repository myvari/IEEE-CIS-Fraud-Detection initialized!

# Data Split

In [4]:
df_transaction = pd.read_csv(DATA_DIR / 'train_transaction.csv')
df_identity = pd.read_csv(DATA_DIR / 'train_identity.csv')

# df_transaction.columns = df_transaction.columns.str.replace("id_", "id-", regex=False)
# df_identity.columns = df_identity.columns.str.replace("id_", "id-", regex=False)

In [5]:
from sklearn.model_selection import train_test_split
df = pd.merge(df_transaction, df_identity, on='TransactionID', how='left')


df = df.sort_values("TransactionDT").reset_index(drop=True)

X = df.drop(columns=['isFraud'])
y = df['isFraud']

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )   

split_index = int(len(df) * 0.8) # time-aware split

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

# Data Cleaning

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin

class DataCleaner(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        drop_transaction_id=True,
        missing_threshold=0.95,
        drop_constant=True,
        drop_high_cardinality_cols=["DeviceInfo"]
    ):
        self.drop_transaction_id = drop_transaction_id
        self.missing_threshold = missing_threshold
        self.drop_constant = drop_constant
        self.drop_high_cardinality_cols = drop_high_cardinality_cols

    def fit(self, X, y=None):
        X = X.copy()
        X.columns = X.columns.str.replace("id-", "id_", regex=False)

        self.columns_to_drop_ = []

        if self.drop_transaction_id and "TransactionID" in X.columns:
            self.columns_to_drop_.append("TransactionID")

        drop_high_cardinality_cols = (
            self.drop_high_cardinality_cols
            if self.drop_high_cardinality_cols is not None
            else []
        )

        # drop high cardinality columns if selected
        for col in drop_high_cardinality_cols:
            if col in X.columns:
                self.columns_to_drop_.append(col)

        # drop columns if missingness above threshold
        if self.missing_threshold is not None:
            missing_rates = X.isna().mean()
            high_missing_cols = missing_rates[missing_rates > self.missing_threshold].index.tolist()
            self.columns_to_drop_.extend(high_missing_cols)

        # drop constant columns
        if self.drop_constant:
            constant_cols = [
                col for col in X.columns
                if X[col].nunique(dropna=False) <= 1
            ]
            self.columns_to_drop_.extend(constant_cols)

        self.columns_to_drop_ = sorted(set(self.columns_to_drop_))

        return self


    def transform(self, X):
        X = X.copy()
        X.columns = X.columns.str.replace("id-", "id_", regex=False)

        X = X.replace([np.inf, -np.inf], np.nan) # handle infinite values if any

        X = X.drop(columns=self.columns_to_drop_, errors="ignore")

        return X

    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

In [30]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
import numpy as np

def make_tree_preprocessor():
    num_pipeline_tree = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    cat_pipeline_tree = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("ordinal", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipeline_tree, make_column_selector(dtype_include=np.number)),
            ("cat", cat_pipeline_tree, make_column_selector(dtype_include=object)),
        ],
        verbose_feature_names_out=False
    )

# Feature Engineering

In [8]:

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        add_identity_feature=True,
        add_amount_features=True,
        add_time_features=True,
        add_email_groups=True,
    ):
        self.add_identity_feature = add_identity_feature
        self.add_amount_features = add_amount_features
        self.add_time_features = add_time_features
        self.add_email_groups = add_email_groups

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X.columns = X.columns.str.replace("id-", "id_", regex=False)

        if self.add_identity_feature:
            if "id_01" in X.columns: # defensive check since some versions of the dataset have inconsistent id column naming
                X["has_identity"] = X["id_01"].notna().astype(int)
            else:
                X["has_identity"] = 0

        # transaction amount features
        if self.add_amount_features and "TransactionAmt" in X.columns:
            X["TransactionAmt_decimal"] = (X["TransactionAmt"] - np.floor(X["TransactionAmt"]))

        if self.add_time_features and "TransactionDT" in X.columns:
            seconds_in_hour = 3600
            seconds_in_day = seconds_in_hour * 24

            transaction_hour = (X["TransactionDT"] // seconds_in_hour) % 24
            transaction_day = X["TransactionDT"] // seconds_in_day
            transaction_week = transaction_day // 7

            X["transaction_day"] = transaction_day
            X["transaction_week"] = transaction_week

            # cyclic encoding for time features (better reflecting of their nature)
            day_of_month = transaction_day % 31
            X["transaction_day_of_month_sin"] = np.sin(2 * np.pi * day_of_month / 31)
            X["transaction_day_of_month_cos"] = np.cos(2 * np.pi * day_of_month / 31)

            weekday = transaction_day % 7
            X["transaction_weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
            X["transaction_weekday_cos"] = np.cos(2 * np.pi * weekday / 7)

            X["transaction_hour_sin"] = np.sin(2 * np.pi * transaction_hour / 24)
            X["transaction_hour_cos"] = np.cos(2 * np.pi * transaction_hour / 24)


        if self.add_email_groups:
            for col in ["P_emaildomain", "R_emaildomain"]:
                if col in X.columns:
                    X[f"{col}_group"] = X[col].apply(self._group_email_domain)

        X = X.replace([np.inf, -np.inf], np.nan)

        return X
    
    def _group_email_domain(self, value):
        if pd.isna(value):
            return "missing"

        value = str(value).lower()

        if "gmail" in value:
            return "gmail"
        elif "yahoo" in value:
            return "yahoo"
        elif value in ["hotmail.com", "outlook.com", "live.com", "msn.com"]:
            return "microsoft"
        elif "icloud" in value or "me.com" in value or "mac.com" in value:
            return "apple"
        elif "anonymous" in value:
            return "anonymous"
        elif "aol" in value:
            return "aol"
        else:
            return "other"
    
    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

# Feature Selection

In [9]:
class NumericCorrelationReducer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95, verbose=True):
        self.threshold = threshold
        self.verbose = verbose

    def fit(self, X, y=None):
        X_df = X.copy()

        self.feature_names_ = X_df.columns.tolist()
        self.numeric_features_ = X_df.select_dtypes(include=[np.number]).columns.tolist()
        self.non_numeric_features_ = [
            col for col in self.feature_names_
            if col not in self.numeric_features_
        ]

        if len(self.numeric_features_) <= 1:
            self.removed_features_ = []
            self.kept_features_ = self.feature_names_
            return self

        numeric_df = X_df[self.numeric_features_]

        corr_matrix = numeric_df.corr().abs()

        mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
        upper = corr_matrix.where(mask)

        to_remove = set()

        for col in upper.columns:
            correlated_features = upper[upper[col] > self.threshold].index.tolist()

            for other in correlated_features:
                col_score = corr_matrix[col].mean()
                other_score = corr_matrix[other].mean()

                if col_score > other_score:
                    to_remove.add(col)
                else:
                    to_remove.add(other)

        self.removed_features_ = sorted(to_remove)
        self.numeric_kept_features_ = [
            feature for feature in self.numeric_features_
            if feature not in self.removed_features_
        ]
        self.kept_features_ = [
            feature for feature in self.feature_names_
            if feature not in self.removed_features_
        ]

        if self.verbose:
            print(f"[[Numeric Correlation]] Removed {len(self.removed_features_)} features: {self.removed_features_}")
            print(f"[[Numeric Correlation]] Kept {len(self.kept_features_)} features: {self.numeric_kept_features_}")

        return self

    def transform(self, X):
        X_df = X.copy()
        X_df.columns = self.feature_names_

        return X_df[self.kept_features_]

    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

In [10]:
class InformationValueSelector(BaseEstimator, TransformerMixin): # before OHE to avoid dimensionality explosion and sparse matrix issues
    def __init__(
        self,
        threshold=0.01,
        n_bins=10,
        max_unique_for_categorical=100,
        min_bin_size=50,
        verbose=True
    ):
        self.threshold = threshold
        self.n_bins = n_bins
        self.max_unique_for_categorical = max_unique_for_categorical
        self.min_bin_size = min_bin_size
        self.verbose = verbose

    def fit(self, X, y):
        X_df = X.copy()
        y_series = pd.Series(y).reset_index(drop=True)
        X_df = X_df.reset_index(drop=True)

        self.feature_names_ = X_df.columns.tolist()
        self.iv_scores_ = {}

        for col in self.feature_names_:
            try:
                iv = self._calculate_feature_iv(X_df[col], y_series)
            except Exception:
                iv = 0.0

            self.iv_scores_[col] = iv

        self.iv_scores_df_ = (
            pd.DataFrame({
                "feature": list(self.iv_scores_.keys()),
                "iv": list(self.iv_scores_.values())
            })
            .sort_values("iv", ascending=False)
            .reset_index(drop=True)
        )

        self.selected_features_ = self.iv_scores_df_.loc[self.iv_scores_df_["iv"] >= self.threshold, "feature"].tolist()

        if len(self.selected_features_) == 0:
            best_feature = self.iv_scores_df_.iloc[0]["feature"] # select the best feature, avoid empty feature set
            self.selected_features_ = [best_feature]

        if self.verbose:
            print(f"[IV] Selected {len(self.selected_features_)} / {len(self.feature_names_)} features")
            print("[IV] Top features:")
            print(self.iv_scores_df_.head(20))

        return self

    def transform(self, X):
        X_df = X.copy()
        X_df.columns = self.feature_names_

        return X_df[self.selected_features_]

    def _calculate_feature_iv(self, feature, y):
        prepared_feature = self._prepare_feature(feature) # convert to categorical with appropriate binning

        df = pd.DataFrame({"feature": prepared_feature, "target": y})

        grouped = df.groupby("feature", dropna=False)["target"].agg(events="sum", total="count")

        grouped["non_events"] = grouped["total"] - grouped["events"]

        # -tiny bins to reduce instability
        grouped = grouped[grouped["total"] >= self.min_bin_size]

        if grouped.shape[0] <= 1:
            return 0.0

        total_events = grouped["events"].sum()
        total_non_events = grouped["non_events"].sum()

        if total_events == 0 or total_non_events == 0:
            return 0.0

        eps = 0.5 # smoothing factor

        grouped["event_dist"] = (grouped["events"] + eps) / (total_events + eps * grouped.shape[0])
        grouped["non_event_dist"] = (grouped["non_events"] + eps) / (total_non_events + eps * grouped.shape[0])

        grouped["woe"] = np.log(grouped["event_dist"] / grouped["non_event_dist"])
        grouped["iv_component"] = (grouped["event_dist"] - grouped["non_event_dist"]) * grouped["woe"]

        return grouped["iv_component"].sum()

    def _prepare_feature(self, feature):
        feature = feature.copy()

        if pd.api.types.is_numeric_dtype(feature):
            unique_count = feature.nunique(dropna=True)

            if unique_count > self.n_bins:
                try:
                    return pd.qcut(feature, q=self.n_bins, duplicates="drop").astype(str).fillna("missing")
                except Exception:
                    return feature.fillna(-999999)
            else:
                return feature.fillna(-999999)

        else:
            feature = feature.astype("object").where(feature.notna(), "missing")
            unique_count = feature.nunique(dropna=True)

            if unique_count > self.max_unique_for_categorical:
                top_values = feature.value_counts().head(self.max_unique_for_categorical).index
                feature = feature.where(feature.isin(top_values), "rare") # rare category for infrequent

            return feature.astype(str)

    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

In [11]:
class NearZeroVarianceReducer(BaseEstimator, TransformerMixin):
    def __init__(self, min_unique=2, max_dominant_ratio=0.995, verbose=True):
        self.min_unique = min_unique
        self.max_dominant_ratio = max_dominant_ratio
        self.verbose = verbose

    def fit(self, X, y=None):
        X_df = X.copy()
        self.feature_names_ = X_df.columns.tolist()
        self.removed_features_ = []

        for col in self.feature_names_:
            value_counts = X_df[col].value_counts(dropna=False, normalize=True)

            if len(value_counts) < self.min_unique:
                self.removed_features_.append(col)
            elif value_counts.iloc[0] >= self.max_dominant_ratio:
                self.removed_features_.append(col)

        self.kept_features_ = [
            col for col in self.feature_names_
            if col not in self.removed_features_
        ]

        if self.verbose:
            print(f"[Near-zero variance] Removed {len(self.removed_features_)} features")
            if len(self.removed_features_) <= 50:
                print(self.removed_features_)
            else:
                print(self.removed_features_[:50], "...")

        return self

    def transform(self, X):
        X_df = X.copy()
        return X_df[self.kept_features_]

    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

# Full Pipline

In [31]:
preprocessing_pipeline = Pipeline([
    ("cleaner", DataCleaner()),
    ("feature_engineer", FeatureEngineer()),
    ("near_zero_variance", NearZeroVarianceReducer(verbose=True)),
    ("preprocessor", make_tree_preprocessor()),
])

preprocessing_pipeline_iv = Pipeline([
    ("cleaner", DataCleaner()),
    ("feature_engineer", FeatureEngineer()),
    ("near_zero_variance", NearZeroVarianceReducer(verbose=True)),
    ("iv_selector", InformationValueSelector(verbose=True)),
    ("preprocessor", make_tree_preprocessor()),
])

preprocessing_pipeline_corr = Pipeline([
    ("cleaner", DataCleaner()),
    ("feature_engineer", FeatureEngineer()),
    ("near_zero_variance", NearZeroVarianceReducer(verbose=True)),
    ("num_corr_reduce", NumericCorrelationReducer(verbose=True)),
    ("preprocessor", make_tree_preprocessor()),
])

In [ ]:
import mlflow
import joblib

mlflow.set_experiment("ieee_fraud_preprocessing")

if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(run_name="RandomForest_Preprocessing_Baseline"):

    mlflow.log_param("cleaner_drop_transaction_id", True)
    mlflow.log_param("cleaner_missing_threshold", 0.95)
    mlflow.log_param("cleaner_drop_constant", True)
    mlflow.log_param("cleaner_drop_high_cardinality_cols", "DeviceInfo")

    mlflow.log_param("add_identity_feature", True)
    mlflow.log_param("add_amount_features", True)
    mlflow.log_param("add_time_features", True)
    mlflow.log_param("add_email_groups", True)
    mlflow.log_param("log_transforms_used", False)
    mlflow.log_param("skewness_based_features_used", False)

    mlflow.log_param("num_impute_strategy", "median")
    mlflow.log_param("num_missing_indicator", True)
    mlflow.log_param("cat_impute_strategy", "constant_missing")
    mlflow.log_param("categorical_encoder", "OrdinalEncoder")
    mlflow.log_param("ordinal_handle_unknown", "use_encoded_value")
    mlflow.log_param("ordinal_unknown_value", -1)
    mlflow.log_param("scaler", "None")

    mlflow.log_param("near_zero_variance_used", True)
    mlflow.log_param("near_zero_variance_max_dominant_ratio", 0.995)

    mlflow.log_param("iv_selector_used", False)
    mlflow.log_param("numeric_corr_reducer_used", False)

    preprocessing_pipeline.fit(X_train, y_train)

    joblib.dump(preprocessing_pipeline, "rf_preprocessing_pipeline_baseline.joblib")
    mlflow.log_artifact("rf_preprocessing_pipeline_baseline.joblib")

    preprocessing_run_id = mlflow.active_run().info.run_id
    print(f"Preprocessing run id: {preprocessing_run_id}")

[Near-zero variance] Removed 12 features
['C3', 'V107', 'V108', 'V111', 'V113', 'V117', 'V118', 'V119', 'V120', 'V121', 'V122', 'V305']
Preprocessing run id: 7b6ef17a9d11499393b396d35912444d
🏃 View run RandomForest_Preprocessing_Baseline at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/7b6ef17a9d11499393b396d35912444d
🧪 View experiment at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


In [32]:
if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(run_name="RandomForest_Preprocessing_IV"):

    mlflow.log_param("cleaner_drop_transaction_id", True)
    mlflow.log_param("cleaner_missing_threshold", 0.95)
    mlflow.log_param("cleaner_drop_constant", True)
    mlflow.log_param("cleaner_drop_high_cardinality_cols", "DeviceInfo")

    mlflow.log_param("add_identity_feature", True)
    mlflow.log_param("add_amount_features", True)
    mlflow.log_param("add_time_features", True)
    mlflow.log_param("add_email_groups", True)
    mlflow.log_param("log_transforms_used", False)
    mlflow.log_param("skewness_based_features_used", False)

    mlflow.log_param("num_impute_strategy", "median")
    mlflow.log_param("num_missing_indicator", True)
    mlflow.log_param("cat_impute_strategy", "constant_missing")
    mlflow.log_param("categorical_encoder", "OrdinalEncoder")
    mlflow.log_param("ordinal_handle_unknown", "use_encoded_value")
    mlflow.log_param("ordinal_unknown_value", -1)
    mlflow.log_param("scaler", "None")

    mlflow.log_param("near_zero_variance_used", True)
    mlflow.log_param("near_zero_variance_max_dominant_ratio", 0.995)

    mlflow.log_param("iv_selector_used", True)
    mlflow.log_param("iv_selector_threshold", 0.01)
    mlflow.log_param("iv_n_bins", 10)
    mlflow.log_param("iv_max_unique_for_categorical", 100)
    mlflow.log_param("iv_min_bin_size", 50)

    mlflow.log_param("numeric_corr_reducer_used", False)

    preprocessing_pipeline_iv.fit(X_train, y_train)

    joblib.dump(preprocessing_pipeline_iv, "rf_preprocessing_pipeline_iv.joblib")
    mlflow.log_artifact("rf_preprocessing_pipeline_iv.joblib")

    rf_iv_preprocessing_run_id = mlflow.active_run().info.run_id
    print(f"RF IV preprocessing run id: {rf_iv_preprocessing_run_id}")

[Near-zero variance] Removed 12 features
['C3', 'V107', 'V108', 'V111', 'V113', 'V117', 'V118', 'V119', 'V120', 'V121', 'V122', 'V305']
[IV] Selected 367 / 422 features
[IV] Top features:
   feature        iv
0     V258  0.918307
1     V257  0.894767
2     V189  0.774238
3     V187  0.762457
4     V201  0.762154
5     V188  0.751724
6     V200  0.750811
7     V199  0.749528
8     V246  0.749493
9     V244  0.749077
10    V243  0.746679
11    V190  0.742765
12    V242  0.735650
13    V186  0.716952
14    V245  0.655815
15    V259  0.641158
16    V171  0.635981
17    V229  0.635286
18    V230  0.633518
19    V218  0.617973
RF IV preprocessing run id: 97c9e524bf68414ca11d438eebdde0ab
🏃 View run RandomForest_Preprocessing_IV at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/97c9e524bf68414ca11d438eebdde0ab
🧪 View experiment at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4


In [33]:
if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(run_name="RandomForest_Preprocessing_Correlation"):

    mlflow.log_param("cleaner_drop_transaction_id", True)
    mlflow.log_param("cleaner_missing_threshold", 0.95)
    mlflow.log_param("cleaner_drop_constant", True)
    mlflow.log_param("cleaner_drop_high_cardinality_cols", "DeviceInfo")

    mlflow.log_param("add_identity_feature", True)
    mlflow.log_param("add_amount_features", True)
    mlflow.log_param("add_time_features", True)
    mlflow.log_param("add_email_groups", True)
    mlflow.log_param("log_transforms_used", False)
    mlflow.log_param("skewness_based_features_used", False)

    mlflow.log_param("num_impute_strategy", "median")
    mlflow.log_param("num_missing_indicator", True)
    mlflow.log_param("cat_impute_strategy", "constant_missing")
    mlflow.log_param("categorical_encoder", "OrdinalEncoder")
    mlflow.log_param("ordinal_handle_unknown", "use_encoded_value")
    mlflow.log_param("ordinal_unknown_value", -1)
    mlflow.log_param("scaler", "None")

    mlflow.log_param("near_zero_variance_used", True)
    mlflow.log_param("near_zero_variance_max_dominant_ratio", 0.995)

    mlflow.log_param("iv_selector_used", False)
    mlflow.log_param("numeric_corr_reducer_used", True)
    mlflow.log_param("numeric_corr_threshold", 0.98)

    preprocessing_pipeline_corr.fit(X_train, y_train)

    joblib.dump(preprocessing_pipeline_corr, "rf_preprocessing_pipeline_corr.joblib")
    mlflow.log_artifact("rf_preprocessing_pipeline_corr.joblib")

    rf_corr_preprocessing_run_id = mlflow.active_run().info.run_id
    print(f"RF correlation preprocessing run id: {rf_corr_preprocessing_run_id}")

[Near-zero variance] Removed 12 features
['C3', 'V107', 'V108', 'V111', 'V113', 'V117', 'V118', 'V119', 'V120', 'V121', 'V122', 'V305']
[[Numeric Correlation]] Removed 99 features: ['C1', 'C10', 'C11', 'C12', 'C14', 'C2', 'C4', 'C6', 'C8', 'D2', 'D4', 'D6', 'D7', 'TransactionDT', 'V10', 'V131', 'V136', 'V137', 'V145', 'V15', 'V150', 'V151', 'V154', 'V155', 'V156', 'V160', 'V163', 'V18', 'V182', 'V187', 'V191', 'V193', 'V199', 'V203', 'V205', 'V206', 'V207', 'V21', 'V215', 'V216', 'V218', 'V222', 'V225', 'V242', 'V248', 'V249', 'V250', 'V254', 'V256', 'V266', 'V269', 'V27', 'V272', 'V276', 'V278', 'V280', 'V29', 'V291', 'V292', 'V295', 'V298', 'V302', 'V31', 'V312', 'V320', 'V321', 'V323', 'V326', 'V327', 'V33', 'V332', 'V333', 'V334', 'V336', 'V338', 'V339', 'V42', 'V48', 'V50', 'V51', 'V57', 'V58', 'V59', 'V63', 'V64', 'V69', 'V71', 'V72', 'V73', 'V79', 'V81', 'V85', 'V89', 'V90', 'V92', 'V93', 'V94', 'has_identity', 'transaction_day']
[[Numeric Correlation]] Kept 323 features: ['Tran

In [14]:
X_train_transformed = preprocessing_pipeline.transform(X_train)
X_valid_transformed = preprocessing_pipeline.transform(X_test)

print(X_train_transformed.shape)
print(X_valid_transformed.shape)

(472432, 788)
(118108, 788)


# Random Forest Baseline

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)
from sklearn.base import clone

mlflow.set_experiment("RandomForest_Training")

cv = TimeSeriesSplit(n_splits=3)

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "neg_log_loss": "neg_log_loss",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

param_grid = [
    # shallow likely underfitting
    {"n_estimators": 50, "max_depth": 6, "min_samples_leaf": 50, "min_samples_split": 50, "max_features": "sqrt", "class_weight": "balanced_subsample"},

    # medium complexity
    {"n_estimators": 100, "max_depth": 10, "min_samples_leaf": 30, "min_samples_split": 30, "max_features": "sqrt", "class_weight": "balanced_subsample"},
    {"n_estimators": 150, "max_depth": 10, "min_samples_leaf": 20, "min_samples_split": 20, "max_features": "sqrt", "class_weight": "balanced_subsample"},

    # deeper but regularized
    {"n_estimators": 200, "max_depth": 18, "min_samples_leaf": 10, "min_samples_split": 5, "max_features": "sqrt", "class_weight": "balanced_subsample"},

    # unbalanced version
    {"n_estimators": 200, "max_depth": 14, "min_samples_leaf": 10, "min_samples_split": 10, "max_features": "sqrt", "class_weight": None}

    # # unbalanced, slightly deeper than current unbalanced winner
    # {"n_estimators": 200,"max_depth": 12,"min_samples_leaf": 10, "min_samples_split": 10,"max_features": "sqrt", "class_weight": None},
    # {"n_estimators": 200,"max_depth": 14,"min_samples_leaf": 5, "min_samples_split": 10,"max_features": "sqrt", "class_weight": None},
    # {"n_estimators": 200,"max_depth": 16,"min_samples_leaf": 10, "min_samples_split": 10,"max_features": "sqrt", "class_weight": None},
    # {"n_estimators": 300,"max_depth": 16,"min_samples_leaf": 10, "min_samples_split": 10,"max_features": "sqrt", "class_weight": None},
]

param_grid_extra = [
    # unbalanced, slightly deeper than current unbalanced winner
    {"n_estimators": 200,"max_depth": 12,"min_samples_leaf": 10, "min_samples_split": 10,"max_features": "sqrt", "class_weight": None},
    {"n_estimators": 200,"max_depth": 14,"min_samples_leaf": 5, "min_samples_split": 10,"max_features": "sqrt", "class_weight": None},
    {"n_estimators": 200,"max_depth": 16,"min_samples_leaf": 10, "min_samples_split": 10,"max_features": "sqrt", "class_weight": None},
    {"n_estimators": 300,"max_depth": 16,"min_samples_leaf": 10, "min_samples_split": 10,"max_features": "sqrt", "class_weight": None},
]

results = []

for params in param_grid_extra: #param_grid
    run_name = (
        f"RF_depth{params['max_depth']}_"
        f"split{params['min_samples_split']}_"
        f"leaf{params['min_samples_leaf']}_"
        f"trees{params['n_estimators']}_"
        f"features{params['max_features']}_"
        f"{'balanced' if params['class_weight'] else 'unbalanced'}"
    )

    with mlflow.start_run(run_name=run_name):

        mlflow.set_tag("model_type", "RandomForestClassifier")
        mlflow.set_tag("training_stage", "full_pipeline_timeseries_cv")
        mlflow.set_tag("validation_type", "TimeSeriesSplit")
        mlflow.set_tag("preprocessing_run_id", preprocessing_run_id)

        mlflow.log_params(params)
        mlflow.log_param("cv_n_splits", 3)
        mlflow.log_param("main_metric", "roc_auc")

        model = RandomForestClassifier(
            **params,
            random_state=42,
            n_jobs=-1
        )

        print(f"\nRunning {run_name}...")

        cv_results = cross_validate(
            model,
            X_train_transformed,
            y_train,
            cv=cv,
            scoring=scoring,
            return_train_score=True,
            n_jobs=1,
            error_score="raise"
        )

        train_roc_auc = cv_results["train_roc_auc"].mean()
        val_roc_auc = cv_results["test_roc_auc"].mean()

        train_avg_precision = cv_results["train_average_precision"].mean()
        val_avg_precision = cv_results["test_average_precision"].mean()

        train_log_loss = -cv_results["train_neg_log_loss"].mean()
        val_log_loss = -cv_results["test_neg_log_loss"].mean()

        mlflow.log_metric("cv_train_roc_auc", train_roc_auc)
        mlflow.log_metric("cv_val_roc_auc", val_roc_auc)
        mlflow.log_metric("cv_train_average_precision", train_avg_precision)
        mlflow.log_metric("cv_val_average_precision", val_avg_precision)
        mlflow.log_metric("cv_train_log_loss", train_log_loss)
        mlflow.log_metric("cv_val_log_loss", val_log_loss)
        mlflow.log_metric("cv_val_f1", cv_results["test_f1"].mean())
        mlflow.log_metric("cv_val_precision", cv_results["test_precision"].mean())
        mlflow.log_metric("cv_val_recall", cv_results["test_recall"].mean())
        mlflow.log_metric("roc_auc_gap", train_roc_auc - val_roc_auc)
        mlflow.log_metric("log_loss_gap", val_log_loss - train_log_loss)

        print("Fitting full pipeline on full training split...")
        model.fit(X_train_transformed, y_train)

        valid_proba = model.predict_proba(X_valid_transformed)[:, 1]
        valid_preds = model.predict(X_valid_transformed)

        holdout_roc_auc = roc_auc_score(y_test, valid_proba)
        holdout_avg_precision = average_precision_score(y_test, valid_proba)
        holdout_log_loss = log_loss(y_test, valid_proba)

        mlflow.log_metric("holdout_roc_auc", holdout_roc_auc)
        mlflow.log_metric("holdout_average_precision", holdout_avg_precision)
        mlflow.log_metric("holdout_log_loss", holdout_log_loss)
        mlflow.log_metric("holdout_f1", f1_score(y_test, valid_preds, zero_division=0))
        mlflow.log_metric("holdout_precision", precision_score(y_test, valid_preds, zero_division=0))
        mlflow.log_metric("holdout_recall", recall_score(y_test, valid_preds, zero_division=0))


        model_path = f"{run_name}.joblib"
        joblib.dump(model, model_path)
        mlflow.log_artifact(model_path)

        results.append({
            "run_name": run_name,
            **params,
            "cv_train_roc_auc": train_roc_auc,
            "cv_val_roc_auc": val_roc_auc,
            "holdout_roc_auc": holdout_roc_auc,
            "holdout_average_precision": holdout_avg_precision,
            "holdout_log_loss": holdout_log_loss,
            "roc_auc_gap": train_roc_auc - val_roc_auc
        })

        print(
            f"{run_name} | "
            f"CV AUC={val_roc_auc:.4f} | "
            f"Holdout AUC={holdout_roc_auc:.4f} | "
            f"AP={holdout_avg_precision:.4f} | "
            f"LogLoss={holdout_log_loss:.4f} | "
            f"Gap={train_roc_auc - val_roc_auc:.4f}"
        )

results_df = pd.DataFrame(results).sort_values("holdout_roc_auc", ascending=False)
results_df


Running RF_depth12_split10_leaf10_trees200_featuressqrt_unbalanced...
Fitting full pipeline on full training split...
RF_depth12_split10_leaf10_trees200_featuressqrt_unbalanced | CV AUC=0.8592 | Holdout AUC=0.8553 | AP=0.4094 | LogLoss=0.1071 | Gap=0.0272
🏃 View run RF_depth12_split10_leaf10_trees200_featuressqrt_unbalanced at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/c64767a58c5141989e11db1fbc5990df
🧪 View experiment at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4

Running RF_depth14_split10_leaf5_trees200_featuressqrt_unbalanced...
Fitting full pipeline on full training split...
RF_depth14_split10_leaf5_trees200_featuressqrt_unbalanced | CV AUC=0.8653 | Holdout AUC=0.8613 | AP=0.4204 | LogLoss=0.1055 | Gap=0.0379
🏃 View run RF_depth14_split10_leaf5_trees200_featuressqrt_unbalanced at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/8e6f722360ce40f1bab805998a48842c
🧪 View experiment 

,run_name,n_estimators,max_depth,min_samples_leaf,min_samples_split,max_features,class_weight,cv_train_roc_auc,cv_val_roc_auc,holdout_roc_auc,holdout_average_precision,holdout_log_loss,roc_auc_gap
2,RF_depth16_split10_leaf10_trees200_featuressqr...,200,16,10,10,sqrt,None,0.914159,0.868839,0.865670,0.426943,0.104399,0.045320
3,RF_depth16_split10_leaf10_trees300_featuressqr...,300,16,10,10,sqrt,None,0.914130,0.869320,0.865588,0.426989,0.104451,0.044810
1,RF_depth14_split10_leaf5_trees200_featuressqrt...,200,14,5,10,sqrt,None,0.903228,0.865330,0.861265,0.420362,0.105490,0.037898
0,RF_depth12_split10_leaf10_trees200_featuressqr...,200,12,10,10,sqrt,None,0.886429,0.859228,0.855305,0.409352,0.107127,0.027200


# Random Forest with Numerical Correlation Filtering

In [38]:
X_train_transformed = preprocessing_pipeline_corr.transform(X_train)
X_valid_transformed = preprocessing_pipeline_corr.transform(X_test)

print(X_train_transformed.shape)
print(X_valid_transformed.shape)

(472432, 602)
(118108, 602)


In [39]:
cv = TimeSeriesSplit(n_splits=2)

results_corr = []

for params in param_grid_extra: #param_grid
    run_name = (
        f"RF_CORR_depth{params['max_depth']}_"
        f"leaf{params['min_samples_leaf']}_"
        f"trees{params['n_estimators']}_"
        f"{'balanced' if params['class_weight'] else 'unbalanced'}"
    )

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("model_type", "RandomForestClassifier")
        mlflow.set_tag("training_stage", "timeseries_cv_on_transformed_data")
        mlflow.set_tag("preprocessing_type", "Correlation_Preprocessing")
        mlflow.set_tag("preprocessing_run_id", rf_corr_preprocessing_run_id)
        mlflow.set_tag("cv_used", True)
        mlflow.set_tag("cv_type", "TimeSeriesSplit")
        mlflow.set_tag("preprocessing_note", "correlation preprocessing fitted once before CV; unsupervised feature selection")

        mlflow.log_params(params)
        mlflow.log_param("cv_n_splits", 2)
        mlflow.log_param("main_metric", "roc_auc")

        model = RandomForestClassifier(
            **params,
            random_state=42,
            n_jobs=-1
        )

        cv_results = cross_validate(
            model,
            X_train_transformed,
            y_train,
            cv=cv,
            scoring=scoring,
            return_train_score=True,
            n_jobs=1,
            error_score="raise"
        )

        train_roc_auc = cv_results["train_roc_auc"].mean()
        val_roc_auc = cv_results["test_roc_auc"].mean()
        train_avg_precision = cv_results["train_average_precision"].mean()
        val_avg_precision = cv_results["test_average_precision"].mean()
        train_log_loss = -cv_results["train_neg_log_loss"].mean()
        val_log_loss = -cv_results["test_neg_log_loss"].mean()

        mlflow.log_metric("cv_train_roc_auc", train_roc_auc)
        mlflow.log_metric("cv_val_roc_auc", val_roc_auc)
        mlflow.log_metric("cv_train_average_precision", train_avg_precision)
        mlflow.log_metric("cv_val_average_precision", val_avg_precision)
        mlflow.log_metric("cv_train_log_loss", train_log_loss)
        mlflow.log_metric("cv_val_log_loss", val_log_loss)
        mlflow.log_metric("cv_val_f1", cv_results["test_f1"].mean())
        mlflow.log_metric("cv_val_precision", cv_results["test_precision"].mean())
        mlflow.log_metric("cv_val_recall", cv_results["test_recall"].mean())
        mlflow.log_metric("roc_auc_gap", train_roc_auc - val_roc_auc)
        mlflow.log_metric("log_loss_gap", val_log_loss - train_log_loss)

        model.fit(X_train_transformed, y_train)

        valid_proba = model.predict_proba(X_valid_transformed)[:, 1]
        valid_preds = model.predict(X_valid_transformed)

        holdout_roc_auc = roc_auc_score(y_test, valid_proba)
        holdout_avg_precision = average_precision_score(y_test, valid_proba)
        holdout_log_loss = log_loss(y_test, valid_proba)

        mlflow.log_metric("holdout_roc_auc", holdout_roc_auc)
        mlflow.log_metric("holdout_average_precision", holdout_avg_precision)
        mlflow.log_metric("holdout_log_loss", holdout_log_loss)
        mlflow.log_metric("holdout_f1", f1_score(y_test, valid_preds, zero_division=0))
        mlflow.log_metric("holdout_precision", precision_score(y_test, valid_preds, zero_division=0))
        mlflow.log_metric("holdout_recall", recall_score(y_test, valid_preds, zero_division=0))

        model_path = f"{run_name}.joblib"
        joblib.dump(model, model_path)
        mlflow.log_artifact(model_path)

        results_corr.append({
            "run_name": run_name,
            "preprocessing": "correlation",
            **params,
            "cv_train_roc_auc": train_roc_auc,
            "cv_val_roc_auc": val_roc_auc,
            "holdout_roc_auc": holdout_roc_auc,
            "holdout_average_precision": holdout_avg_precision,
            "holdout_log_loss": holdout_log_loss,
            "roc_auc_gap": train_roc_auc - val_roc_auc,
        })

        print(
            f"{run_name} | "
            f"CV AUC={val_roc_auc:.4f} | "
            f"Holdout AUC={holdout_roc_auc:.4f} | "
            f"AP={holdout_avg_precision:.4f} | "
            f"LogLoss={holdout_log_loss:.4f} | "
            f"Gap={train_roc_auc - val_roc_auc:.4f}"
        )

results_corr_df = pd.DataFrame(results_corr).sort_values("holdout_roc_auc", ascending=False)
results_corr_df

RF_CORR_depth12_leaf10_trees200_unbalanced | CV AUC=0.8617 | Holdout AUC=0.8553 | AP=0.4094 | LogLoss=0.1071 | Gap=0.0242
🏃 View run RF_CORR_depth12_leaf10_trees200_unbalanced at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/49e51f45acec4ecb978cdfa339091565
🧪 View experiment at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4
RF_CORR_depth14_leaf5_trees200_unbalanced | CV AUC=0.8667 | Holdout AUC=0.8613 | AP=0.4204 | LogLoss=0.1055 | Gap=0.0354
🏃 View run RF_CORR_depth14_leaf5_trees200_unbalanced at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/965ca8364f5b4c5fb30417d08763d91e
🧪 View experiment at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4
RF_CORR_depth16_leaf10_trees200_unbalanced | CV AUC=0.8720 | Holdout AUC=0.8657 | AP=0.4269 | LogLoss=0.1044 | Gap=0.0421
🏃 View run RF_CORR_depth16_leaf10_trees200_unbalanced at: https://dagshub.com/myvari/IEEE-CIS-Fraud-

,run_name,preprocessing,n_estimators,max_depth,min_samples_leaf,min_samples_split,max_features,class_weight,cv_train_roc_auc,cv_val_roc_auc,holdout_roc_auc,holdout_average_precision,holdout_log_loss,roc_auc_gap
2,RF_CORR_depth16_leaf10_trees200_unbalanced,correlation,200,16,10,10,sqrt,None,0.914017,0.871959,0.865670,0.426943,0.104399,0.042058
3,RF_CORR_depth16_leaf10_trees300_unbalanced,correlation,300,16,10,10,sqrt,None,0.914528,0.872439,0.865588,0.426989,0.104451,0.042090
1,RF_CORR_depth14_leaf5_trees200_unbalanced,correlation,200,14,5,10,sqrt,None,0.902158,0.866746,0.861265,0.420362,0.105490,0.035412
0,RF_CORR_depth12_leaf10_trees200_unbalanced,correlation,200,12,10,10,sqrt,None,0.885905,0.861663,0.855305,0.409352,0.107127,0.024243


# Random Forest with IV-selected features

In [40]:
X_train_transformed = preprocessing_pipeline_iv.transform(X_train)
X_valid_transformed = preprocessing_pipeline_iv.transform(X_test)

print(X_train_transformed.shape)
print(X_valid_transformed.shape)

(472432, 682)
(118108, 682)


In [41]:
results_iv = []

for params in param_grid:
    run_name = (
        f"RF_FAST_depth{params['max_depth']}_"
        f"leaf{params['min_samples_leaf']}_"
        f"trees{params['n_estimators']}_"
        f"{'balanced' if params['class_weight'] else 'unbalanced'}"
    )

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("model_type", "RandomForestClassifier")
        mlflow.set_tag("training_stage", "fast_holdout_screening")
        mlflow.set_tag("preprocessing_type", "IV_Preprocessing")
        mlflow.set_tag("preprocessing_run_id", preprocessing_run_id)
        mlflow.set_tag("cv_used", False)

        mlflow.log_params(params)

        model = RandomForestClassifier(
            **params,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train_transformed, y_train)

        valid_proba = model.predict_proba(X_valid_transformed)[:, 1]
        valid_preds = model.predict(X_valid_transformed)

        holdout_roc_auc = roc_auc_score(y_test, valid_proba)
        holdout_avg_precision = average_precision_score(y_test, valid_proba)
        holdout_log_loss = log_loss(y_test, valid_proba)

        mlflow.log_metric("holdout_roc_auc", holdout_roc_auc)
        mlflow.log_metric("holdout_average_precision", holdout_avg_precision)
        mlflow.log_metric("holdout_log_loss", holdout_log_loss)
        mlflow.log_metric("holdout_f1", f1_score(y_test, valid_preds, zero_division=0))
        mlflow.log_metric("holdout_precision", precision_score(y_test, valid_preds, zero_division=0))
        mlflow.log_metric("holdout_recall", recall_score(y_test, valid_preds, zero_division=0))


        joblib.dump(model, f"{run_name}.joblib")
        mlflow.log_artifact(f"{run_name}.joblib")

        results_iv.append({
            "run_name": run_name,
            **params,
            "holdout_roc_auc": holdout_roc_auc,
            "holdout_average_precision": holdout_avg_precision,
            "holdout_log_loss": holdout_log_loss,
        })

        print(
            f"{run_name} | "
            f"AUC={holdout_roc_auc:.4f} | "
            f"AP={holdout_avg_precision:.4f} | "
            f"LogLoss={holdout_log_loss:.4f}"
        )

results_iv_df = pd.DataFrame(results_iv).sort_values("holdout_roc_auc", ascending=False)
results_iv_df

RF_FAST_depth6_leaf50_trees50_balanced | AUC=0.8462 | AP=0.3948 | LogLoss=0.4649
🏃 View run RF_FAST_depth6_leaf50_trees50_balanced at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/32e6ab78da934d0baf94d1bdbffe4ef0
🧪 View experiment at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4
RF_FAST_depth10_leaf30_trees100_balanced | AUC=0.8668 | AP=0.4445 | LogLoss=0.4052
🏃 View run RF_FAST_depth10_leaf30_trees100_balanced at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/7f6546f9560e44d080d88f6d7349e6e2
🧪 View experiment at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4
RF_FAST_depth10_leaf20_trees150_balanced | AUC=0.8654 | AP=0.4426 | LogLoss=0.4054
🏃 View run RF_FAST_depth10_leaf20_trees150_balanced at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/95fb64225d5343b796e9307274209f2a
🧪 View experiment at: https://dagshub.com/myvari/IEEE-

,run_name,n_estimators,max_depth,min_samples_leaf,min_samples_split,max_features,class_weight,holdout_roc_auc,holdout_average_precision,holdout_log_loss
3,RF_FAST_depth18_leaf10_trees200_balanced,200,18,10,5,sqrt,balanced_subsample,0.884452,0.475683,0.275212
4,RF_FAST_depth14_leaf10_trees200_unbalanced,200,14,10,10,sqrt,None,0.867088,0.436857,0.103900
1,RF_FAST_depth10_leaf30_trees100_balanced,100,10,30,30,sqrt,balanced_subsample,0.866834,0.444509,0.405226
2,RF_FAST_depth10_leaf20_trees150_balanced,150,10,20,20,sqrt,balanced_subsample,0.865400,0.442568,0.405449
0,RF_FAST_depth6_leaf50_trees50_balanced,50,6,50,50,sqrt,balanced_subsample,0.846165,0.394835,0.464917


In [42]:
best_params = results_iv_df.iloc[0][[
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "min_samples_split",
    "max_features",
    "class_weight"
]].to_dict()

if pd.isna(best_params["class_weight"]):
    best_params["class_weight"] = None

best_params

{'n_estimators': 200,
 'max_depth': 18,
 'min_samples_leaf': 10,
 'min_samples_split': 5,
 'max_features': 'sqrt',
 'class_weight': 'balanced_subsample'}

In [43]:
cv = TimeSeriesSplit(n_splits=2)

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "neg_log_loss": "neg_log_loss"
}

with mlflow.start_run(run_name="RandomForest_Finalist_TimeSeriesCV"):
    mlflow.set_tag("model_type", "RandomForestClassifier")
    mlflow.set_tag("training_stage", "finalist_timeseries_cv")
    mlflow.set_tag("cv_used", True)
    mlflow.set_tag("cv_type", "TimeSeriesSplit")
    mlflow.set_tag("preprocessing_type", "IV_Preprocessing")
    mlflow.set_tag("preprocessing_run_id", rf_iv_preprocessing_run_id)
    mlflow.set_tag("preprocessing_note", "preprocessing fitted once before CV; no target leakage except IV if IV preprocessing was used")

    mlflow.log_params(best_params)
    mlflow.log_param("cv_n_splits", 2)

    finalist_model = RandomForestClassifier(
        **best_params,
        random_state=42,
        n_jobs=-1
    )

    cv_results = cross_validate(
        finalist_model,
        X_train_transformed,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
        error_score="raise"
    )

    train_auc = cv_results["train_roc_auc"].mean()
    val_auc = cv_results["test_roc_auc"].mean()

    train_ap = cv_results["train_average_precision"].mean()
    val_ap = cv_results["test_average_precision"].mean()

    train_log_loss = -cv_results["train_neg_log_loss"].mean()
    val_log_loss = -cv_results["test_neg_log_loss"].mean()

    mlflow.log_metric("cv_train_roc_auc", train_auc)
    mlflow.log_metric("cv_val_roc_auc", val_auc)
    mlflow.log_metric("cv_train_average_precision", train_ap)
    mlflow.log_metric("cv_val_average_precision", val_ap)
    mlflow.log_metric("cv_train_log_loss", train_log_loss)
    mlflow.log_metric("cv_val_log_loss", val_log_loss)
    mlflow.log_metric("roc_auc_gap", train_auc - val_auc)
    mlflow.log_metric("log_loss_gap", val_log_loss - train_log_loss)

    print("Random Forest Finalist CV Results")
    print("-" * 45)
    print(f"CV Train ROC-AUC: {train_auc:.4f}")
    print(f"CV Val ROC-AUC:   {val_auc:.4f}")
    print(f"CV Val AP:        {val_ap:.4f}")
    print(f"CV Val LogLoss:   {val_log_loss:.4f}")
    print(f"ROC-AUC Gap:      {train_auc - val_auc:.4f}")

Random Forest Finalist CV Results
---------------------------------------------
CV Train ROC-AUC: 0.9811
CV Val ROC-AUC:   0.8762
CV Val AP:        0.5008
CV Val LogLoss:   0.2564
ROC-AUC Gap:      0.1049
🏃 View run RandomForest_Finalist_TimeSeriesCV at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/97586e7023da4131b1a26c3db7accd84
🧪 View experiment at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4


In [44]:
best_model = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

final_pipeline = Pipeline([
    ("preprocessing", clone(preprocessing_pipeline_iv)),
    ("model", best_model)
])

with mlflow.start_run(run_name="RandomForest_Final_FullPipeline"):
    mlflow.set_tag("model_type", "RandomForestClassifier")
    mlflow.set_tag("training_stage", "final_full_pipeline")
    mlflow.set_tag("preprocessing_type", "IV_Preprocessing")
    mlflow.set_tag("main_metric", "holdout_roc_auc")
    mlflow.log_params(best_params)

    print("Fitting final full pipeline...")
    final_pipeline.fit(X_train, y_train)

    print("Generating validation predictions...")
    valid_proba = final_pipeline.predict_proba(X_test)[:, 1]
    valid_preds = final_pipeline.predict(X_test)

    holdout_roc_auc = roc_auc_score(y_test, valid_proba)
    holdout_avg_precision = average_precision_score(y_test, valid_proba)
    holdout_log_loss = log_loss(y_test, valid_proba)
    holdout_f1 = f1_score(y_test, valid_preds, zero_division=0)
    holdout_precision = precision_score(y_test, valid_preds, zero_division=0)
    holdout_recall = recall_score(y_test, valid_preds, zero_division=0)

    mlflow.log_metric("holdout_roc_auc", holdout_roc_auc)
    mlflow.log_metric("holdout_average_precision", holdout_avg_precision)
    mlflow.log_metric("holdout_log_loss", holdout_log_loss)
    mlflow.log_metric("holdout_f1", holdout_f1)
    mlflow.log_metric("holdout_precision", holdout_precision)
    mlflow.log_metric("holdout_recall", holdout_recall)

    model_path = "rf_final_full_pipeline.joblib"
    joblib.dump(final_pipeline, model_path)
    mlflow.log_artifact(model_path)

    print("\nFinal Random Forest Full Pipeline Results")
    print("-" * 55)
    print(f"ROC-AUC:           {holdout_roc_auc:.4f}")
    print(f"Average Precision: {holdout_avg_precision:.4f}")
    print(f"Log Loss:          {holdout_log_loss:.4f}")
    print(f"F1 Score:          {holdout_f1:.4f}")
    print(f"Precision:         {holdout_precision:.4f}")
    print(f"Recall:            {holdout_recall:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, valid_preds))

    print("\nClassification Report:")
    print(classification_report(y_test, valid_preds, zero_division=0))

    final_run_id = mlflow.active_run().info.run_id
    print(f"\nFinal run id: {final_run_id}")

Fitting final full pipeline...
[Near-zero variance] Removed 12 features
['C3', 'V107', 'V108', 'V111', 'V113', 'V117', 'V118', 'V119', 'V120', 'V121', 'V122', 'V305']
[IV] Selected 367 / 422 features
[IV] Top features:
   feature        iv
0     V258  0.918307
1     V257  0.894767
2     V189  0.774238
3     V187  0.762457
4     V201  0.762154
5     V188  0.751724
6     V200  0.750811
7     V199  0.749528
8     V246  0.749493
9     V244  0.749077
10    V243  0.746679
11    V190  0.742765
12    V242  0.735650
13    V186  0.716952
14    V245  0.655815
15    V259  0.641158
16    V171  0.635981
17    V229  0.635286
18    V230  0.633518
19    V218  0.617973
Generating validation predictions...

Final Random Forest Full Pipeline Results
-------------------------------------------------------
ROC-AUC:           0.8845
Average Precision: 0.4757
Log Loss:          0.2752
F1 Score:          0.4012
Precision:         0.3138
Recall:            0.5561

Confusion Matrix:
[[109102   4942]
 [  1804   2

In [4]:
print(mlflow.search_runs(
    experiment_names=["RandomForest_Training"],
    order_by=["metrics.holdout_roc_auc DESC", "metrics.roc_auc_gap ASC", "metrics.holdout_log_loss ASC"]
)[['tags.mlflow.runName', 'metrics.cv_train_roc_auc', 'metrics.cv_val_roc_auc', 'metrics.roc_auc_gap', 'metrics.holdout_roc_auc', 'tags.preprocessing_type']].to_markdown(index=False))

| tags.mlflow.runName                                        |   metrics.cv_train_roc_auc |   metrics.cv_val_roc_auc |   metrics.roc_auc_gap |   metrics.holdout_roc_auc | tags.preprocessing_type   |
|:-----------------------------------------------------------|---------------------------:|-------------------------:|----------------------:|--------------------------:|:--------------------------|
| RF_depthNone_split5_leaf10_trees300_featuressqrt_balanced  |                   0.997242 |                 0.884028 |             0.113214  |                  0.893314 |                           |
| RandomForest_Final_FullPipeline                            |                 nan        |               nan        |           nan         |                  0.884452 | IV_Preprocessing          |
| RF_FAST_depth18_leaf10_trees200_balanced                   |                 nan        |               nan        |           nan         |                  0.884452 | IV_Preprocessing          |
| RF_

In [48]:
import numpy as np

rf_runs = mlflow.search_runs(
    experiment_names=["RandomForest_Training"],
    #filter_string="tags.preprocessing_type = 'Correlation_Preprocessing'",#IV_Preprocessing #Correlation_Preprocessing  #full_pipeline_timeseries_cv\
    #filter_string="tags.training_stage = 'fast_holdout_screening'"
    order_by=["metrics.holdout_roc_auc DESC"]
)

cols = [
    "tags.mlflow.runName",
    "metrics.cv_train_roc_auc",
    "metrics.cv_val_roc_auc",
    "metrics.roc_auc_gap",
    "metrics.holdout_roc_auc",
    "metrics.holdout_average_precision",
    "metrics.holdout_log_loss",
    "tags.preprocessing_type",
    "tags.training_stage",
]

available_cols = [col for col in cols if col in rf_runs.columns]

rf_df = rf_runs[available_cols].copy()

rf_df = rf_df.rename(columns={
    "tags.mlflow.runName": "run_name",
    "metrics.cv_train_roc_auc": "cv_train_auc",
    "metrics.cv_val_roc_auc": "cv_val_auc",
    "metrics.roc_auc_gap": "auc_gap",
    "metrics.holdout_roc_auc": "holdout_auc",
    "metrics.holdout_average_precision": "holdout_ap",
    "metrics.holdout_log_loss": "holdout_log_loss",
    "tags.preprocessing_type": "preprocessing",
    "tags.training_stage": "training_stage",
})

rf_df["preprocessing"] = rf_df["preprocessing"].replace("", np.nan)

rf_df["preprocessing"] = rf_df["preprocessing"].fillna("Baseline_Preprocessing")

rf_df = rf_df.sort_values("holdout_auc", ascending=False).reset_index(drop=True)

#print(rf_df.to_markdown(index=False))

In [50]:
best_by_preprocessing = (
    rf_df
    .sort_values("holdout_auc", ascending=False)
    .groupby("preprocessing", as_index=False)
    .first()
    [[
        "preprocessing",
        "run_name",
        "cv_train_auc",
        "cv_val_auc",
        "auc_gap",
        "holdout_auc",
        "holdout_ap",
        "holdout_log_loss"
    ]]
    .sort_values("holdout_auc", ascending=False)
)

print(best_by_preprocessing.to_markdown(index=False))

| preprocessing             | run_name                                                  |   cv_train_auc |   cv_val_auc |   auc_gap |   holdout_auc |   holdout_ap |   holdout_log_loss |
|:--------------------------|:----------------------------------------------------------|---------------:|-------------:|----------:|--------------:|-------------:|-------------------:|
| Baseline_Preprocessing    | RF_depthNone_split5_leaf10_trees300_featuressqrt_balanced |       0.997242 |     0.884028 |  0.113214 |      0.893314 |     0.487518 |           0.192474 |
| IV_Preprocessing          | RandomForest_Final_FullPipeline                           |       0.98108  |     0.876215 |  0.104865 |      0.884452 |     0.475683 |           0.275212 |
| Correlation_Preprocessing | RF_CORR_depth18_leaf10_trees200_balanced                  |       0.976543 |     0.866129 |  0.110414 |      0.876401 |     0.454638 |           0.282879 |
